# Calculate effective coverage by quintile and scenario

Also by age, sex, and pregnancy status (though coverage will not vary by pregnancy status due to a lack of data).

This is similar to what the pregnancy simulation does at the individual level, but using groups instead.
It can be shared between multiplication models that do not incorporate individual heterogeneity.

In [1]:
import pandas as pd

In [2]:
location = "india"
vehicle = "rice"
fortificant = "iron"

In [3]:
# Parameters
location = "nigeria"
fortificant = "iron"
vehicle = "bouillon"


In [4]:
results_dir = f'../results/{fortificant}/{vehicle}'

In [5]:
full_coverage_probability = pd.read_csv(f'{results_dir}/baseline_fortification/full_coverage/{location}.csv')
full_coverage_probability = full_coverage_probability.set_index([c for c in full_coverage_probability.columns if c != 'value']).value
full_coverage_probability

vehicle_name  wealth_quintile
bouillon      lowest             0.519211
              second             0.584098
              middle             0.638105
              fourth             0.669371
              highest            0.699797
Name: value, dtype: float64

In [6]:
any_coverage_probability = pd.read_csv(f'{results_dir}/baseline_fortification/any_coverage/{location}.csv')
any_coverage_probability = any_coverage_probability.set_index([c for c in any_coverage_probability.columns if c != 'value']).value
any_coverage_probability

vehicle_name  wealth_quintile
bouillon      lowest             0.519211
              second             0.584098
              middle             0.638105
              fourth             0.669371
              highest            0.699797
Name: value, dtype: float64

In [7]:
partial_coverage_mean = pd.read_csv(f'{results_dir}/baseline_fortification/partial_coverage_amount/mean/{location}.csv')
partial_coverage_mean = partial_coverage_mean.set_index([c for c in partial_coverage_mean.columns if c != 'value']).value
partial_coverage_mean

wealth_quintile  vehicle_name
lowest           bouillon        0
second           bouillon        0
middle           bouillon        0
fourth           bouillon        0
highest          bouillon        0
Name: value, dtype: int64

In [8]:
current_coverage = full_coverage_probability + (any_coverage_probability - full_coverage_probability) * partial_coverage_mean
current_coverage

vehicle_name  wealth_quintile
bouillon      lowest             0.519211
              second             0.584098
              middle             0.638105
              fourth             0.669371
              highest            0.699797
Name: value, dtype: float64

In [9]:
scenarios = {
    "india": ["intervention"],
    "nigeria": ["intervention"],
    "ethiopia": ["intervention_25_nrv", "intervention_100_nrv"],
}[location]

In [10]:
fortifiability = pd.read_csv(f'../results/{vehicle}/vehicle_consumption/fortifiability/{location}.csv')
fortifiability = fortifiability.set_index([c for c in fortifiability.columns if c != 'value']).value
fortifiability

vehicle_name  wealth_quintile  sex   
bouillon      lowest           Female    0.965732
              second           Female    0.978593
              middle           Female    0.980847
              fourth           Female    0.988844
              highest          Female    0.983773
              lowest           Male      0.965732
              second           Male      0.978593
              middle           Male      0.980847
              fourth           Male      0.988844
              highest          Male      0.983773
Name: value, dtype: float64

In [11]:
import pathlib

for scenario in scenarios:
    intervention_coverage = pd.read_csv(f'{results_dir}/{scenario}/intervention_fortification/any_coverage/{location}.csv')
    intervention_coverage = intervention_coverage.set_index([c for c in intervention_coverage.columns if c != 'value']).value
    target_coverage = intervention_coverage * fortifiability
    display(target_coverage)
    assert (target_coverage > current_coverage.reindex_like(target_coverage)).all()
    # Not all coverage is effective -- this is as a proportion of coverage!
    effectiveness = pd.read_csv(f'{results_dir}/{scenario}/intervention_fortification/effectiveness/{location}.csv')
    effectiveness = effectiveness.set_index([c for c in effectiveness.columns if c != 'value']).value
    effective_intervention_coverage = target_coverage * effectiveness
    path = f'{results_dir}/{scenario}/intervention_fortification/effective_coverage/{location}.csv'
    pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
    effective_intervention_coverage.reset_index().to_csv(path, index=False)


vehicle_name  wealth_quintile  sex   
bouillon      lowest           Female    0.772586
              second           Female    0.782875
              middle           Female    0.784677
              fourth           Female    0.791075
              highest          Female    0.787018
              lowest           Male      0.772586
              second           Male      0.782875
              middle           Male      0.784677
              fourth           Male      0.791075
              highest          Male      0.787018
Name: value, dtype: float64

In [12]:
# Not all coverage is effective -- this is as a proportion of coverage!
effectiveness = pd.read_csv(f'{results_dir}/baseline_fortification/effectiveness/{location}.csv')
effectiveness = effectiveness.set_index([c for c in effectiveness.columns if c != 'value']).value

In [13]:
effective_baseline_coverage = current_coverage * effectiveness
effective_baseline_coverage

vehicle_name  wealth_quintile
bouillon      fourth             0.535497
              highest            0.559838
              lowest             0.415369
              middle             0.510484
              second             0.467278
Name: value, dtype: float64

In [14]:
path = f'{results_dir}/baseline_fortification/effective_coverage/{location}.csv'
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
effective_baseline_coverage.reset_index().to_csv(path, index=False)